# ဦးစားပေးအဖွဲ့ဝင် Middleware နှင့် ဟိုတယ်ဘတ်ခ္ကင်

ဒီ notebook က Microsoft Agent Framework ကိုသုံးတဲ့ **function-based middleware** ကို ပြသပါတယ်။ Conditional workflow နမူနာကို အခြေခံပြီး ဦးစားပေးအဖွဲ့ဝင်တွေကို အထူးအခွင့်အရေး ပေးတဲ့ middleware အလွှာကို တိုးထည့်ထားပါတယ်။

## သင်ယူနိုင်မယ့်အကြောင်းအရာများ:
1. **Function-Based Middleware**: function ရလဒ်တွေကို ရိုက်နှိပ်ပြင်ဆင်ခြင်း
2. **Context Access**: လည်ပတ်ပြီးနောက် `context.result` ကို ဖတ်ခြင်းနှင့် ပြင်ဆင်ခြင်း
3. **လုပ်ငန်းနည်းဗျူဟာ အကောင်အထည်ဖော်ခြင်း**: ဦးစားပေးအဖွဲ့ဝင်အကျိုးခံစားခွင့်များ
4. **ရလဒ်ပြောင်းလဲခြင်း**: အသုံးပြုသူအခြေအနေ အပေါ်မူတည်ပြီး function ရလဒ်များကို ပြောင်းလဲခြင်း
5. **တူညီWorkflow၊ မတူညီတဲ့ရလဒ်များ**: Middleware က ထိန်းချုပ်ထားသော အပြုအမူပြောင်းလဲမှုများ

## Middleware ပါသော Workflow အဆောက်အအုံ:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Conditional Workflow နှင့်ကွဲပြားချက်အဓိက:

**Middleware မပါသည့်အခြေအနေ** (14-conditional-workflow.ipynb):
- ပြယ်ရီမှာ အခန်းရှာမတွေ့ → alternative_agent ဆီ ပြောင်းလဲပို့ဆောင်

**Middleware ပါသည့်အခြေအနေ** (ဒီ notebook):
- ပုံမှန်အသုံးပြုသူ + ပြယ်ရီ → အခန်းမရှိ → alternative_agent ဆီ ပြောင်းလဲပို့ဆောင်
- ဦးစားပေးအသုံးပြုသူ + ပြယ်ရီ → 🌟 Middleware override လုပ်! → အခန်းရှိ → booking_agent ဆီ ပို့ပေး

## လိုအပ်ချက်များ:
- Microsoft Agent Framework ထည့်သွင်းထားရမည်
- Conditional workflows နားလည်မှုရှိရမည် (14-conditional-workflow.ipynb ကိုကြည့်ပါ)
- GitHub token သို့မဟုတ် OpenAI API key ရှိရမည်
- Middleware patterns များ အခြေခံနားလည်မှုရှိရမည်


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## အဆင့် ၁: ဖွဲ့စည်းထားသော output များအတွက် Pydantic မော်ဒယ်များသတ်မှတ်ခြင်း

ဤမော်ဒယ်များသည် agent များပြန်မည့် **schema** ကိုသတ်မှတ်သည်။ middleware က availability ရလဒ်ကိုပြောင်းလဲသောအခါကိုစစ်ဆေးရန် `priority_override` ကွက်တိကိုထည့်သွင်းထားသည်။


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## အဆင့် ၂: ဦးစားပေး အဖွဲ့ဝင်များ ဒေတာဘေ့စ် သတ်မှတ်ခြင်း  

ဒီ ဒီမိုအတွက် ‌ဦးစားပေး အဖွဲ့ဝင် ဒေတာဘေ့စ်ကို အတုယူဖော်ပြပါမည်။ ထုတ်လုပ်မှုတွင်တော့ တကယ့် ဒေတာဘေ့စ် သို့မဟုတ် API ကို ဆွဲယူမည်ဖြစ်သည်။  

**ဦးစားပေး အဖွဲ့ဝင်များ:**
- `alice@example.com` - VIP အဖွဲ့ဝင်
- `bob@example.com` - Premium အဖွဲ့ဝင်  
- `priority_user` - စမ်းသပ် အကောင့်


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## အဆင့် ၃: ဟိုတယ်မှာယူခြင်းကိရိယာကို ဖန်တီးခြင်း

ကန့်သတ်ထားသော workflow လိုပဲ ဖြစ်ပေမယ့် အခုတော့ middleware နဲ့ ကာကွယ်မိပါပြီ!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## အဆင့် ၄: 🌟 ဦးစားပေး စစ်ဆေးမှု Middleware (အဓိက လက္ခဏာ!)

ဒီဟာက ဒီ notebook ရဲ့ **အဓိက လုပ်ဆောင်ချက်** ဖြစ်တယ်။ Middleware မှာ:

1. hotel_booking function ဖုန်းခေါ်ချက်ကို **အတားအဆီး ပြုလုပ်တယ်**
2. `next(context)` ကိုခေါ်ပြီး function ကို ပုံမှန်အတိုင်း **အကောင်အထည်ဖော်တယ်**
3. `context.result` ထဲကရလဒ်ကို **စစ်ဆေးတယ်**
4. အသုံးပြုသူက ဦးစားပေးဖြစ်ပြီး အခန်းမရရှိနိုင်ရင်ရလဒ်ကို **ကျော်လွှားတယ်**
5. ပြင်ဆင်ပြီးတဲ့ရလဒ်ကို ပြန်လည် **agent ဆီ ပြန်ပေးတယ်**

**အဓိက ပုံစံ:**
```python
async def my_middleware(context, next):
    await next(context)  # လုပ်ဆောင်ချက်ကို အကောင်အထည်ဖော်ပါ
    # ယခု context.result တွင် လုပ်ဆောင်ချက်၏ ထွက်လာသည့် အချက်အလက်များ ပါဝင်သည်
    if some_condition:
        context.result = new_value  # အစားထိုးပါ!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## အဆင့် ၅: လမ်းကြောင်းသတ်မှတ်ခြင်းအတွက် အခြေအနေ Function များ သတ်မှတ်ပါ

အခြေအနေ Function များသည် conditional workflow တွင် အသုံးပြုသော Function များနှင့် အတူတူဖြစ်ပြီး၊ structured output ကို သုံးသပ်၍ လမ်းကြောင်းကို သတ်မှတ်သည်။ 


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## အဆင့် ၆: စိတ်ကြိုက်ပြသသူ အပြောင်းအလဲ ဖန်တီးပါ

ယခင်ကဲ့သို့ မည်သည့်အပြောင်းအလဲမှ မဟုတ်သေးဘဲ - နောက်ဆုံးလုပ်ဆောင်မှု အလုပ်စဉ်ရလဒ်ကို ပြသပါသည်။


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## အဆင့် ၇: ပတ်ဝန်းကျင် အပြောင်းအလဲများအား ထည့်သွင်းရန်

LLM client (Microsoft Foundry / Azure OpenAI) ကို စီစစ်ဖွဲ့စည်းပါ။


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## အဆင့် ၈: Middleware ဖြင့် AI Agents ဖန်တီးခြင်း

**အဓိက ကွာခြားချက်:** availability_agent ကို ဖန်တီးရာတွင် `middleware` ပေရာမီတာကို ပေးပို့သည်!

ဤနည်းဖြင့် priority_check_middleware ကို agent ၏ function invocation pipeline တွင် ထည့်သွင်းသည်။


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## အဆင့် ၉: လုပ်ငန်းစဉ်ကို တည်ဆောက်ပါ

အရင်ကလည်း တူညီတဲ့ လုပ်ငန်းစဉ်ဖွဲ့စည်းမှု - ရရှိနိုင်မှုအပေါ် မူတည်ပြီး အခြေအနေဆိုင်ရာ လမ်းကြောင်းခွဲခြားမှု။


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## အဆင့် ၁၀: စမ်းသပ်မှုကိစ္စ ၁ - ပဲရစ်ရှိ ပုံမှန်အသုံးပြုသူ (Override မရှိ)

ပုံမှန်အသုံးပြုသူတစ်ဦးသည် ပဲရစ် အတွက် ရွှေ့ဆိုင်းကြိုးစား → အခန်းမရှိ → alternative_agent သို့ ပြောင်းရွှေ့


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## အဆင့် ၁၁: စမ်းသပ်မှုကိစ္စ ၂ - 🌟 ပရီယိုရิตี้ အသုံးပြုသူ ပါရီ(ပြုပြင်ခြင်းနှင့်!)

ပရီယိုရิตี้ အဖွဲ့ဝင်တစ်ဦးသည် ပါရီကို စီစဉ်ရန်ကြိုးစားသည် → ညအခန်းများ မရှိသေး → 🌟 Middleware က override လုပ်သည်! → booking_agent ဆီသို့ ဦးတည်သည်

**ဤသည်မှာ middleware အင်အား၏ အဓိကပြသချက်ဖြစ်သည်!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## အဆင့် 12: စမ်းသပ်မှုကိစ္စ 3 - စတိုက်သော့လမ်းရှိ ဦးစားပေး အသုံးပြုသူ (တစ်ခြားတယ်)

ဦးစားပေး အသုံးပြုသူသည် စတိုက်သော့လမ်းကြိုး → အခန်းများ ရနိုင်သည် → အစားထိုးရန် မလိုအပ်တော့ → booking_agent သို့ လမ်းကြောင်း ရွေးချယ်သည်

ဤသည်သည် middleware သည် လိုအပ်သောအခါတွင်သာ လုပ်ဆောင်သည်ကို ပြသသည်!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## အဓိကယူဆချက်များနှင့် Middleware အယူခံများ

### ✅ သင်ယူခဲ့တာတွေ:

#### **1. Function-Based Middleware Pattern**

Middleware က async function ရိုးရှင်းတစ်ခုကို အသုံးပြုပြီး function ခေါ်ဆိုမှုများကို ထိန်းသိမ်းတယ်။

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ဖင်ရှင် လုပ်ဆောင်မှုမပြုမီ
    print("Intercepting...")
    
    # ဖင်ရှင်ကို လုပ်ဆောင်ပါ
    await next(context)
    
    # ဖင်ရှင် ပြီးဆုံးပြီးနောက် - ရလဒ်ကို စစ်ဆေးပါ
    if context.result:
        # လိုအပ်လျှင် ရလဒ်ကို သွားပြင်ပါ
        context.result = modified_value
```

#### **2. Context Access and Result Override**

- `context.function` - ခေါ်ဆိုနေတဲ့ function ကိုရယူခြင်း
- `context.arguments` - function arguments များကို ဖတ်ရှုခြင်း
- `context.kwargs` - ထပ်ဆောင်း parameters များကို အသုံးပြုခြင်း
- `await next(context)` - function ကို အလုပ်လုပ်စေခြင်း
- `context.result` - function ရဲ့ ထွက်ရှိမှုကို ဖတ်ရှု/ပြင်ဆင်ခြင်း

#### **3. စီးပွားရေးအခြေအနေဆိုင်ရာ အကောင်အထည်ဖေါ်ခြင်း**

ကျွန်ုပ်တို့ middleware က priority member အကျိုးခံစားခွင့်များကို အကောင်အထည်ဖေါ်သည်။
- **ပုံမှန်အသုံးပြုသူများ**: ပြင်ဆင်ချက်မရှိ၊ စံလမ်းကြောင်း
- **Priority အသုံးပြုသူများ**: "အသုံးပြုနိုင်မှုမရှိ" ကို "အသုံးပြုနိုင်" အဖြစ်ကာကွယ်ပြောင်းလဲခြင်း
- **အခြေအနေဆိုင်ရာ logic**: လိုအပ်သည့်အခါသာ override လုပ်ခြင်း

#### **4. Workflow တူ၊ အကျိုးသက်ရောက်မှု မတူ**

Middleware ၏အားကောင်းခြင်း:
- ✅ Workflow ဖွဲ့စည်းမှု မပြောင်းလဲမှု
- ✅ Tool function မပြောင်းလဲမှု
- ✅ အခြေအနေဆိုင်ရာ routing logic မပြောင်းလဲမှု
- ✅ Middleware ထဲကနေသာ → အပြုအမူ ကွဲပြားခြင်း!

### 🚀 အမှန်တကယ် အသုံးချမှုများ:

1. **VIP/Premium Features**
   - premium user များအတွက် rate limit များကို override လုပ်ခြင်း
   - အရင်းအမြစ်များကို priority ဖြင့် access ပြုခြင်း
   - premium features ကို dynamic unlock ပေးခြင်း

2. **A/B စမ်းသပ်မှု**
   - အသုံးပြုသူများကို အမျိုးမျိုးသော implementation များသို့ ဦးတည်စေခြင်း
   - သတ်မှတ်ထားသောအသုံးပြုသူများနှင့် features အသစ် စမ်းသပ်ခြင်း
   - features များကို ဆက်တိုက် gradually ထည့်သွင်းခြင်း

3. **လုံခြုံရေး & ကိုက်ညီမှု**
   - function ခေါ်ဆိုမှုများကို စစ်ဆေးခြင်း
   - sensitive operations များကို ကန့်သတ်ခြင်း
   - စီးပွားရေး စည်းကမ်းများကို စည်းမျဉ်းတင်ခြင်း

4. **စွမ်းဆောင်ရည် မြှင့်တင်ခြင်း**
   - သတ်မှတ်ထားသော user များအတွက် အောက်ဆက်ထုတ်လုပ်မှုများကို ချထားခြင်း
   - ဝယ်သာသော operations များကို ကာလကာလအလိုက် ကျော်လွှားခြင်း
   - dynamic resource များ အစီအစဉ်ခြင်း

5. **အမှားများကို ကိုင်တွယ်ခြင်း & ပြန်လည်ကြိုးစားခြင်း**
   - အမှားနေရာများကို ဖမ်းဆီးပြီး သန့်ရှင်းစွာ တုံ့ပြန်ခြင်း
   - ပြန်လည်ကြိုးစားမှု logic ကို အကောင်အထည်ဖော်ခြင်း
   - အခြား implement များသို့ fallback ပြုလုပ်ခြင်း

6. **မှတ်တမ်းတင်ခြင်း & စောင့်ကြည့်ခြင်း**
   - function လုပ်ငန်းချိန်များကို ချိန်ဆ
   - parameters နှင့် ရလဒ်များကို မှတ်တမ်းတင်ခြင်း
   - အသုံးပြုမှုပုံစံများကို စောင့်ကြည့်ခြင်း

### 🔑 Decorator များနှင့် ကွဲပြားချက်များ:

| လက္ခဏာ | Decorator | Middleware |
|---------|-----------|------------|
| **အကွာအဝေး** | တစ်ခုတည်းသော function | agent ထဲရှိ functions တွေအားလုံး |
| **တုံ့ပြန်နိုင်မှု** | သတ်မှတ်ချက်မှာ သေချာရှိ | အလုပ်လုပ်စဉ်အတွင်း dynamic ဖြစ်နိုင် |
| **Context** | ကန့်သတ်ထားသည် | agent context အပြည့်အစုံ |
| **ဖွဲ့စည်းမှု** | decorator များစုံ | Middleware pipeline |
| **Agent ကိုသိရှိမှု** | မရှိ | ရှိ (agent အခြေအနေကို access ပြုနိုင်) |

### 📚 Middleware ကို ဘယ်အခါ သုံးရမလဲ:

✅ **Middleware သည် အောက်ပါအခါ သုံးပါ:**
- အသုံးပြုသူ/session အခြေအနေ အပေါ် အပြုအမူ ပြောင်းလဲရန် လိုအပ်သည့်အခါ
- functions များစွာတွင် logic အကောင်အထည်ဖော်ရန် လိုအပ်သည့်အခါ
- agent-level context ကို ရယူရန် လိုအပ်သည့်အခါ
- cross-cutting issues (logging, auth, စသဖြင့်) ကို အကောင်အထည်ဖော်သည့်အခါ

❌ **Middleware မသုံးသင့်သောအခါ:**
- ရိုးရှင်းသော input စစ်ဆေးခြင်း (Pydantic ကို သုံးပါ)
- function-specific logic (function အတွင်းမှာသာ ထားပါ)
- တစ်ကြိမ်ပြင်ဆင်ခြင်းများ (function ကို တိုက်ရိုက်ပြောင်းလဲပါ)

### 🎓 အဆင့်မြင့်ပုံစံများ:

```python
# မျိုးစုံရှိသော middleware များ (အကောင်အထည်ဖော်မှုအစဉ်အတိုင်းအရေးကြီးသည်!)
middleware=[
    logging_middleware,      # အရင်ဆုံး မှတ်တမ်းတင်သည်
    auth_middleware,         # ပြီးမှ ခွင့်ပြုချက်စစ်ဆေးသည်
    cache_middleware,        # ပြီးမှ cache စစ်ဆေးသည်
    rate_limit_middleware,   # ပြီးမှ အရေအတွက်ကန့်သတ်ချက်များ
    priority_check_middleware  # နောက်ဆုံး priority စစ်ဆေးမှု
]

# အခြေအနေမူတည် middleware အကောင်အထည်ဖော်မှု
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ရလဒ် ပြောင်းလဲသည်
    else:
        # အကောင်အထည်ဖော်မှုကို လုံးဝ ကျော်လွှားသည်
        context.result = cached_value
```

### 🔗 ဆက်စပ်အယူခံများ:

- **Agent Middleware**: agent.run() ခေါ်ဆိုမှုများကို ဆွဲယူဆိုင်းငံ့ခြင်း
- **Function Middleware**: tool function ခေါ်ဆိုမှုများကို ဆွဲယူဆိုင်းငံ့ခြင်း (ကျွန်ုပ်တို့ သုံးခဲ့သည်)
- **Middleware Pipeline**: Middleware များကို အမည်ရင်းစဉ်ဖက်စည်းကြိတ်ခြင်း
- **Context Propagation**: Middleware ချိတ်ဆက်မှုတွင် အခြေအနေ ဖြန့်ဝေရန်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
